# Off_targets 

In [20]:
import random
import math
import numpy as np
from itertools import combinations
import time

In [21]:
def generate_off_targets(sequence_len, mismatches):
    off_targets = []

    for mismatch_positions in combinations(range(sequence_len), mismatches):
        seq = 0
        for pos in mismatch_positions:
            seq ^= (1 << pos)  # flip to 0
        off_targets.append(seq)

    return off_targets

In [22]:
total_off_targets = generate_off_targets(20, 4)

In [23]:
for off_target in total_off_targets:
    print(f"{off_target:020b}")
print(len(total_off_targets))

00000000000000001111
00000000000000010111
00000000000000100111
00000000000001000111
00000000000010000111
00000000000100000111
00000000001000000111
00000000010000000111
00000000100000000111
00000001000000000111
00000010000000000111
00000100000000000111
00001000000000000111
00010000000000000111
00100000000000000111
01000000000000000111
10000000000000000111
00000000000000011011
00000000000000101011
00000000000001001011
00000000000010001011
00000000000100001011
00000000001000001011
00000000010000001011
00000000100000001011
00000001000000001011
00000010000000001011
00000100000000001011
00001000000000001011
00010000000000001011
00100000000000001011
01000000000000001011
10000000000000001011
00000000000000110011
00000000000001010011
00000000000010010011
00000000000100010011
00000000001000010011
00000000010000010011
00000000100000010011
00000001000000010011
00000010000000010011
00000100000000010011
00001000000000010011
00010000000000010011
00100000000000010011
01000000000000010011
1000000000000

# Metrics

### Coverage

In [24]:
masks = [
0b11111011010000000000,
0b00000100101111110000,
0b11000100100000001111,
0b00111000001100001110,
0b00000011010011101001,
0b10000011001100010101,
0b01111000000011010001,
0b00100110110000110010,
0b11000000011111000010,
0b01011101101000100000,
0b10010000010010111100,
0b00001111000011000110,
0b00100100111100001001,
0b01100001000100111010,
0b10101000100001100101,
0b00011010010100010011,
0b10010011101001001000,
0b01001010011000101100,
0b00110101000010000111,
0b01010001110101000100,
0b10001100000001101011,
0b11001010100010011000,
0b01010000001000110111,
0b10101001001110100000,
0b01110110110010000000,
0b00001000111001010110,
0b10010001011000101001,
0b10110111100100000000,
]

In [25]:
def check_coverage(mask_set, off_targets):
    covered = 0

    for off_target in off_targets:
        valid = False
        for mask in mask_set:
            if (off_target & mask) == 0:
                covered += 1
                valid = True
                break
        # if not valid:
            # print(f"off_target: {off_target:020b}")

    return covered / len(off_targets)


def increment_coverage(mask, coverage, remaining_off_targets):
        for off_target in remaining_off_targets:
            if (off_target & mask) == 0:
                coverage += 1
                
        return coverage / len(total_off_targets)

def remove_covered_off_targets(mask_set, remaining_off_targets):
    new_remaining = []

    for off_target in remaining_off_targets:
        covered = False
        for mask in mask_set:
            if (off_target & mask) == 0:
                covered = True
                break

        if not covered:
            new_remaining.append(off_target)

    return new_remaining

In [26]:
result = check_coverage(masks, total_off_targets)
result
    
    

1.0

### Overlap

In [27]:
def check_overlap(mask_set, mask, length):
    overlap_pos = []
    max_overlap = max((mask & m).bit_count() for m in mask_set)
    return max_overlap, overlap_pos

In [28]:
def get_overlap_complexity(mask_set, length):
    overlap_complexity = 0

    if len(mask_set) < 1:
        return 0
        
    for i, m1 in enumerate(mask_set):
        for j, m2 in enumerate(mask_set):
            if j <= i:
                continue
                
            for shift in range(-(length - 1), length):

                if shift >= 0:
                    shifted = m2 >> shift
                else:
                    shifted = m2 << (-shift)

                overlap = (m1 & shifted).bit_count()

                overlap_complexity += 2 ** overlap
                
    return overlap_complexity


def increment_overlap_complexity(mask_set, mask, length):
    overlap_complexity = 0
    test_set = mask_set + [mask]

    if len(test_set) < 1:
        return 0
        
    for m in mask_set:
        for shift in range(-(length - 1), length):
            if shift >= 0:
                shifted = m >> shift
            else:
                shifted = m << (-shift)

            overlap = (mask & shifted).bit_count()
            overlap_complexity += 2 ** overlap
                
    return overlap_complexity

### Position Penalties

In [124]:
mismatch_penalties = [0.0, 0.0, 0.014, 0.0, 0.0, 0.395, 0.317, 0.0, 0.389, 0.079, 0.445, 0.508, 0.613, 0.851, 0.732, 0.828, 0.615, 0.804, 0.685, 0.583]
max_position_penalty = sum(mismatch_penalties)
print(max_position_penalty)

def get_position_score(mask, mismatch_penalties):
    postition_penalty_score = 0
    for i, penalty in enumerate(mismatch_penalties):
        postition_penalty_score += (mask >> i & 1) * penalty
    return postition_penalty_score
        

7.858


# Mask Generation

In [125]:
def generate_mask(weight, length):
    pos = range(length)
    mask = 0
    # remaining_pos = set(pos) - set(overlap_pos)
    positions = random.sample(pos, weight)
    
    for pos in positions:
        mask |= (1 << pos)
    return mask

def get_neighbour(mask, length):
    for i in random.sample(range(length), 4):
        mask ^= 1 << i
    return mask

def get_neighbour_same_weight(mask, length):
    ones = [i for i in range(length) if mask & (1 << i)]
    zeros = [i for i in range(length) if not (mask & (1 << i))]

    off_bit = random.choice(ones)
    on_bit = random.choice(zeros)

    new_mask = mask

    # turn one off
    new_mask ^= (1 << off_bit)

    # turn one on
    new_mask ^= (1 << on_bit)

    return new_mask
            

In [126]:
result = generate_mask(8, 20)
print(f"mask: {result:020b}")

result = get_neighbour(result, 20)
print(f"neighbour mask: {result:020b}")

mask: 00010101000101001101
neighbour mask: 00010001000110001100


### Set Generation

In [127]:
def generate_neighbour_set(mask_set, weight, length, max_masks):
    new_set = mask_set.copy()

    option = random.random()

    # mutate one mask
    if option < 0.8 and len(new_set) > 0:
        idx = random.randrange(len(new_set))
        new_set[idx] = get_neighbour_same_weight(new_set[idx], length)

    # replace one mask
    elif option < 0.9 and len(new_set) > 0:
        idx = random.randrange(len(new_set))
        new_set[idx] = generate_mask(weight, length)

    # add mask
    elif option < 0.95 and len(new_set) < max_masks:
        new_set.append(generate_mask(weight, length))

    #remove mask
    elif option < 1 and len(new_set) > 1:
        idx = random.randrange(len(new_set))
        del new_set[idx]



    return new_set

# Algorithm

In [128]:
max_overlap = (
    len(mask_set)
    * (len(mask_set) - 1)
    / 2
    * weight
)

NameError: name 'weight' is not defined

In [134]:
def evaluate_mask_set(
    mask_set,
    coverage_weight,
    overlap_weight,
    position_weight,
    specificity_weight
):
    coverage = check_coverage(mask_set, total_off_targets)

    max_overlap = 0

    for i in range(len(mask_set)):
        for j in range(i + 1, len(mask_set)):
    
            wa = mask_set[i].bit_count()
            wb = mask_set[j].bit_count()
    
            max_overlap += min(wa, wb)

    overlap = get_overlap_complexity(mask_set, 20)
    overlap_norm = overlap / (max_overlap * 20)

    position_penalty = sum(
        get_position_score(mask, mismatch_penalties)
        for mask in mask_set
    )
    position_penalty_norm = position_penalty / (max_position_penalty * len(mask_set))
    
    weights = [m.bit_count() for m in mask_set]
    avg_weight_norm = sum(weights) / (20 * len(weights))
    # min_weight_norm = min(weights) / length

    # print(coverage)
    # print(overlap_norm)
    # print(avg_weight_norm)
    # print(position_penalty_norm)

    score = (
        coverage_weight * coverage
        - overlap_weight * overlap_norm
        - position_weight * position_penalty_norm
        + specificity_weight * avg_weight_norm
    )

    return score

In [135]:
def find_mask_set(length, num_masks=100, weight=12, coverage_weight=1, overlap_weight=1, position_weight=1, specificity_weight=1, coverage_threshold=0.75, weight_count=100, search_num=300):
    current_set = [
        generate_mask(weight, length)
        for _ in range(num_masks)
    ]
    
    current_score = evaluate_mask_set(
        current_set,
        coverage_weight,
        overlap_weight,
        position_weight,
        specificity_weight
    )

    best_set = current_set.copy()
    best_score = current_score

    T0 = 1.0
    alpha = 0.995
    count = 0

    for iteration in range(search_num):

        T = max(0.001, T0 * (alpha ** iteration))

        candidate_set = generate_neighbour_set(
            current_set,
            weight,
            length,
            num_masks
        )

        candidate_score = evaluate_mask_set(
            candidate_set,
            coverage_weight,
            overlap_weight,
            position_weight,
            specificity_weight
        )

        diff = candidate_score - current_score

        accept = False

        if diff > 0:
            accept = True
        else:
            probability = math.exp(diff / T)

            if random.random() < probability:
                accept = True

        if accept:
            current_set = candidate_set
            current_score = candidate_score

        if current_score > best_score:
            best_set = current_set.copy()
            best_score = current_score
            count = 0
            print(best_score)

            if check_coverage(best_set, total_off_targets) > coverage_threshold:
                break
        else:
            count += 1

        if count == weight_count:
            if weight > 4:
                weight -= 1
                count = 0


    return best_set
    

In [136]:
start_time = time.perf_counter()
mask_set = find_mask_set(
    length = 20, 
    num_masks = 10, 
    weight = 8, 
    coverage_weight = 1,
    overlap_weight = 1, 
    position_weight = 0.2,
    specificity_weight = 1,
    coverage_threshold = 0.9, 
    weight_count = 10000,
    search_num = 1000000
)
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(elapsed_time)

-0.18227350322454716
-0.16261630679161232
-0.15215215653179337
-0.14077642330502993
-0.13414834086773975
-0.12151502595563868
-0.12010012386177304
-0.11379652039648025
-0.10271077908241077
-0.09975047143119936
-0.09711594084035313
-0.07699537052723682
-0.076864002692967
-0.07673166006120158
-0.07166041338505641
-0.0676642622975076
-0.06488522035829813
-0.06361098755139705
-0.061493937628544715
-0.06012972795937671
-0.05461674206326367
-0.05429296756768731
-0.04481278996039356
-0.029398299104775316
-0.023550440555498153
-0.020195291545328675
-0.019503521747341268
-0.01674348600629877
-0.016620182604840994
-0.009106145717899583
-0.004835155800810065
-0.0030552717731477386
-0.0019437381075141924
-0.0017731321295267888
0.00094196880683034
0.0029028932059710355
0.0030996731588377457
0.003306883127526028
0.0036688130597779534
0.004397130923741721
0.005014772225930053
0.005435328998915578
0.008408284064855054
0.008550778839887974
0.009357286472835735
0.011287339110806038
0.013765075598407783


In [137]:
for m in mask_set:
    print(f"mask: {m:020b}")
result = check_coverage(mask_set, total_off_targets)
print(result)

mask: 10000000011001100010
mask: 01010100100000011100
mask: 10100000000111000101
mask: 10000010101000110001
mask: 01101100000000010011
mask: 01001001011010000100
mask: 00000011010010001010
mask: 10010000100110010010
mask: 00011100001001001001
mask: 01100011000000101000
0.9153766769865841


In [138]:
with open("9_masks.txt", "w") as f:
    for m in mask_set:
        f.write(f"{m:020b}\n")

In [66]:
test_masks = [
0b10001000000000101000,
0b01100010101001010001,
0b10100100100100000111,
0b00000000000000000000
]

In [67]:
check_coverage(test_masks, total_off_targets)

1.0

#### 